In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
import pickle
import matplotlib.pyplot as plt

In [12]:
dataset1 = pd.read_csv("prep.csv")

df2 = dataset1.copy()

df2 = pd.get_dummies(df2, drop_first=True)

In [13]:
df2

,age,bp,al,su,bgr,bu,sc,sod,pot,hrmo,...,pc_normal,pcc_present,ba_present,htn_yes,dm_yes,cad_yes,appet_yes,pe_yes,ane_yes,classification_yes
0,2.000000,76.459948,3.0,0.0,148.112676,57.482105,3.077356,137.528754,4.627244,12.518156,...,False,False,False,False,False,False,True,True,False,True
1,3.000000,76.459948,2.0,0.0,148.112676,22.000000,0.700000,137.528754,4.627244,10.700000,...,True,False,False,False,False,False,True,False,False,True
2,4.000000,76.459948,1.0,0.0,99.000000,23.000000,0.600000,138.000000,4.400000,12.000000,...,True,False,False,False,False,False,True,False,False,True
3,5.000000,76.459948,1.0,0.0,148.112676,16.000000,0.700000,138.000000,3.200000,8.100000,...,True,False,False,False,False,False,True,False,True,True
4,5.000000,50.000000,0.0,0.0,148.112676,25.000000,0.600000,137.528754,4.627244,11.800000,...,True,False,False,False,False,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,51.492308,70.000000,0.0,0.0,219.000000,36.000000,1.300000,139.000000,3.700000,12.500000,...,True,False,False,False,False,False,True,False,False,True
395,51.492308,70.000000,0.0,2.0,220.000000,68.000000,2.800000,137.528754,4.627244,8.700000,...,True,False,False,True,True,False,True,False,True,True
396,51.492308,70.000000,3.0,0.0,110.000000,115.000000,6.000000,134.000000,2.700000,9.100000,...,True,False,False,True,True,False,False,False,False,True
397,51.492308,90.000000,0.0,0.0,207.000000,80.000000,6.800000,142.000000,5.500000,8.500000,...,True,False,False,True,True,False,True,False,True,True


In [14]:
test = SelectKBest(score_func=chi2, k=5)
fit1= test.fit(indep_X,dep_Y)
        # summarize scores       
selectk_features = fit1.transform(indep_X)
selectk_features

array([[1.48112676e+02, 5.74821053e+01, 3.07735602e+00, 3.88689024e+01,
        8.40819113e+03],
       [1.48112676e+02, 2.20000000e+01, 7.00000000e-01, 3.40000000e+01,
        1.23000000e+04],
       [9.90000000e+01, 2.30000000e+01, 6.00000000e-01, 3.40000000e+01,
        8.40819113e+03],
       ...,
       [1.10000000e+02, 1.15000000e+02, 6.00000000e+00, 2.60000000e+01,
        9.20000000e+03],
       [2.07000000e+02, 8.00000000e+01, 6.80000000e+00, 3.88689024e+01,
        8.40819113e+03],
       [1.00000000e+02, 4.90000000e+01, 1.00000000e+00, 5.30000000e+01,
        8.50000000e+03]], shape=(399, 5))

In [16]:
feature_scores = pd.DataFrame({
    'Feature': indep_X.columns,
    'Chi2 Score': fit1.scores_,
    'P-value': fit1.pvalues_
})

feature_scores = feature_scores.sort_values(
    by='Chi2 Score',
    ascending=False
)

display(feature_scores)

,Feature,Chi2 Score,P-value
11,wc,12825.196453,0.000000e+00
4,bgr,2439.902719,0.000000e+00
5,bu,2347.944805,0.000000e+00
6,sc,355.952701,2.142325e-79
10,pcv,326.621491,5.231695e-73
2,al,216.265060,5.903374e-49
9,hrmo,126.154306,2.844805e-29
0,age,114.022622,1.288303e-26
3,su,95.180723,1.737742e-22
21,htn_yes,87.951807,6.706881e-21


In [ ]:
Higher Chi-Square scores indicate stronger association with the target, while lower p-values indicate stronger evidence against independence.

In [17]:
selected_features = fit1.get_support()

print(selected_features)

[False False False False  True  True  True False False False  True  True
 False False False False False False False False False False False False
 False False False]


In [18]:
selected_columns = indep_X.columns[selected_features]

print(selected_columns)

Index(['bgr', 'bu', 'sc', 'pcv', 'wc'], dtype='str')


In [20]:
selectk_features = fit1.transform(indep_X)

print(selectk_features)

[[1.48112676e+02 5.74821053e+01 3.07735602e+00 3.88689024e+01
  8.40819113e+03]
 [1.48112676e+02 2.20000000e+01 7.00000000e-01 3.40000000e+01
  1.23000000e+04]
 [9.90000000e+01 2.30000000e+01 6.00000000e-01 3.40000000e+01
  8.40819113e+03]
 ...
 [1.10000000e+02 1.15000000e+02 6.00000000e+00 2.60000000e+01
  9.20000000e+03]
 [2.07000000e+02 8.00000000e+01 6.80000000e+00 3.88689024e+01
  8.40819113e+03]
 [1.00000000e+02 4.90000000e+01 1.00000000e+00 5.30000000e+01
  8.50000000e+03]]


In [ ]:
This returns the actual data values of the five selected features as a NumPy array.

In [ ]:
Important difference between fit() and transform()
Method	Purpose
fit()	Calculates Chi-Square scores and identifies the best features.
transform()	Returns the dataset containing only the selected features.
fit_transform()	Performs both operations together.

In [ ]:
My recommendation: Use fit_transform() when you want to select features from training data in one step.

If you also need to transform test data, use fit() on training data and then transform() on both training and test data. This avoids data leakage.

In [21]:
test = SelectKBest(score_func=chi2, k=5)

selectk_features = test.fit_transform(indep_X, dep_Y)

selectk_features

array([[1.48112676e+02, 5.74821053e+01, 3.07735602e+00, 3.88689024e+01,
        8.40819113e+03],
       [1.48112676e+02, 2.20000000e+01, 7.00000000e-01, 3.40000000e+01,
        1.23000000e+04],
       [9.90000000e+01, 2.30000000e+01, 6.00000000e-01, 3.40000000e+01,
        8.40819113e+03],
       ...,
       [1.10000000e+02, 1.15000000e+02, 6.00000000e+00, 2.60000000e+01,
        9.20000000e+03],
       [2.07000000e+02, 8.00000000e+01, 6.80000000e+00, 3.88689024e+01,
        8.40819113e+03],
       [1.00000000e+02, 4.90000000e+01, 1.00000000e+00, 5.30000000e+01,
        8.50000000e+03]], shape=(399, 5))

In [23]:
def selectkbest(indep_X,dep_Y,n):
        test = SelectKBest(score_func=chi2, k=n)
        fit1= test.fit(indep_X,dep_Y)
        # summarize scores       
        selectk_features = fit1.transform(indep_X)
        return selectk_features

In [24]:
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)    
        return X_train, X_test, y_train, y_test

In [ ]:
This performs two operations:
- fit() – Calculates the mean and standard deviation of each training feature.
- transform() – Uses those values to standardize the training data.


Why only transform() here?

Because we have already calculated the mean and standard deviation from the training data.

We must use the same training mean and standard deviation to scale the testing data.

We should not use fit_transform() on testing data because it would calculate new statistics from the test data, causing data leakage.


We use fit_transform() on training data to calculate the mean and standard deviation and standardize the training features. For testing data, we use only transform() to apply the same training statistics. This prevents data leakage and ensures proper model evaluation.

In [ ]:
What is Data Leakage?
Data Leakage happens when information from the testing data is accidentally used during the model training process.

In simple words:

We should train our model without allowing it to learn anything from the testing data.

Why? Because testing data should represent unseen data. We use it to check how well our model performs on new data.


Data leakage occurs when information from outside the training dataset, particularly the testing dataset, is used during model training or preprocessing. It can produce misleadingly good evaluation results and reduce the model's ability to generalize to unseen data.

In [32]:
def r2_prediction(regressor,X_test,y_test):
     y_pred = regressor.predict(X_test)
     from sklearn.metrics import r2_score
     r2=r2_score(y_test,y_pred)
     return r2
 
def Linear(X_train,y_train,X_test,y_test):       
        # Fitting K-NN to the Training set
        from sklearn.linear_model import LinearRegression
        regressor = LinearRegression()
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2   
    
def svm_linear(X_train,y_train,X_test,y_test):
                
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'linear')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
    
def svm_NL(X_train,y_train,X_test,y_test):
                
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'rbf')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     

def Decision(X_train,y_train,X_test,y_test):
        
        # Fitting K-NN to the Training setC
        from sklearn.tree import DecisionTreeRegressor
        regressor = DecisionTreeRegressor(random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     

def random(X_train,y_train,X_test,y_test):       
        # Fitting K-NN to the Training set
        from sklearn.ensemble import RandomForestRegressor
        regressor = RandomForestRegressor(n_estimators = 10, random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2 


In [33]:
def selectk_regression(acclin,accsvml,accsvmnl,accdes,accrf): 
    
    dataframe=pd.DataFrame(index=['R2 Score'],columns=['Linear','SVMl','SVMnl','Decision','Random'
                                                                                     ])

    for number,idex in enumerate(dataframe.index):
                
        dataframe.loc[idex, 'Linear'] = acclin[number]
        dataframe.loc[idex, 'SVMl'] = accsvml[number]
        dataframe.loc[idex, 'SVMnl'] = accsvmnl[number]
        dataframe.loc[idex, 'Decision'] = accdes[number]
        dataframe.loc[idex, 'Random'] = accrf[number]
    return dataframe

In [41]:

kbest = selectkbest(indep_X, dep_Y, 12)

acclin = []
accsvml = []
accsvmnl = []
accdes = []
accrf = []

X_train, X_test, y_train, y_test = split_scalar(kbest, dep_Y)

r2_lin = Linear(X_train, y_train, X_test, y_test)
acclin.append(r2_lin)

r2_sl = svm_linear(X_train, y_train, X_test, y_test)
accsvml.append(r2_sl)

r2_NL = svm_NL(X_train, y_train, X_test, y_test)
accsvmnl.append(r2_NL)

r2_d = Decision(X_train, y_train, X_test, y_test)
accdes.append(r2_d)

r2_r = random(X_train, y_train, X_test, y_test)
accrf.append(r2_r)

result = selectk_regression(
    acclin, accsvml, accsvmnl, accdes, accrf
)

result


,Linear,SVMl,SVMnl,Decision,Random
R2 Score,0.672525,0.648182,0.909463,0.826389,0.934028


In [ ]:
K=5

               Linear	SVMl	SVMnl	Decision	Random
R2 Score	0.551985	0.545395	0.749654	0.696181	0.836806



In [ ]:
K=6


                Linear	SVMl	SVMnl	Decision	Random
R2 Score	0.599041	0.586446	0.838962	0.869792	0.897569


In [ ]:
k=7

                Linear	SVMl	SVMnl	Decision	Random
R2 Score	0.657035	0.641906	0.893007	0.826389	0.916233


In [ ]:
k=12

                Linear	SVMl	SVMnl	Decision	Random
R2 Score	0.672525	0.648182	0.909463	0.826389	0.934028


# Random Forest is the best algorithm in this